# SpaGene Training Tutorial

- Loading and preprocessing source/target dataframes  
- Creating cross-validation (CV) splits for genes and samples  
- Build dataloaders
- Training:  
    - Encoder–Decoder
    - Translator  
- Aggregating CV predictions + computing metrics  

In [ ]:
import os, sys, time, glob
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import tifffile as tif

BASE_DIR = Path(os.getcwd())
DATA_DIR = (BASE_DIR / "data/paired_datasets").resolve()

In [2]:
# Project imports
from codes.data_utils import (
    load_dataframe, process_dataframe,
    ExpDataset, CompositeDataset,
    composite_collate_fn,
)
from codes.train_utils import define_trainer

/home/abudhkar/anaconda3/envs/myenv/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
# Parser for opts
import argparse
from pathlib import Path
import os

def get_parser_with_defaults():
    p = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)

    # Train options
    p.add_argument("--loss_type_enc_dec", type=str, default="mse")
    p.add_argument("--loss_type_cyc",     type=str, default="mse")
    p.add_argument("--loss_type_id",      type=str, default="pearson")
    p.add_argument("--loss_type_disc",    type=str, default="hinge")

    p.add_argument("--lambda_cyc",  type=float, default=1.0)
    p.add_argument("--lambda_id",   type=float, default=1.0)
    p.add_argument("--lambda_adv",  type=float, default=0.1)
    p.add_argument("--lambda_disc", type=float, default=0.05)
    p.add_argument("--lambda_gp",   type=float, default=3.0)

    p.add_argument("--interval_enc_dec", type=int, default=1)
    p.add_argument("--interval_id",      type=int, default=1)
    p.add_argument("--interval_cyc",     type=int, default=1)
    p.add_argument("--interval_adv",     type=int, default=1)
    p.add_argument("--interval_disc",    type=int, default=5)

    p.add_argument("--batch_size",      type=int,  default=512)
    p.add_argument("--epochs_enc_dec",  type=int,  default=100)
    p.add_argument("--fix_weight_enc_dec", action="store_true", default=True)
    p.add_argument("--epochs",          type=int,  default=50)

    # split-related
    p.add_argument("--cv_gene",   type=int, default=5)
    p.add_argument("--cv_sample", type=int, default=1)
    p.add_argument("--target_fold_gene",   type=int, default=None)
    p.add_argument("--target_fold_sample", type=int, default=None)

    # others
    p.add_argument("--device",      type=str,   default="cuda")
    p.add_argument("--num_workers", type=int,   default=4)
    p.add_argument("--log_dir",     type=str,   default="results/nano2gse")
    p.add_argument("--r1_every",    type=int,   default=8)
    p.add_argument("--lr_disc",     type=float, default=0.0005)
    p.add_argument("--lr_trans",    type=float, default=0.001)
    p.add_argument("--seed",        type=int,   default=0)

    p.add_argument("--eval_every",     type=int,   default=5)
    p.add_argument("--eval_max_cells", type=int,   default=4096)
    p.add_argument("--early_stop_patience", type=int,   default=10)
    p.add_argument("--early_stop_min_delta", type=float, default=1e-4)

    p.add_argument("--use_amp", action="store_true", default=True)

    # -------------------------
    # Data options
    # -------------------------
    p.add_argument("--domain_source", type=str, default="nanostring")
    p.add_argument("--domain_target", type=str, default="gse")

    # dataframe options
    p.add_argument("--clip_outlier_source", action="store_true", default=True)
    p.add_argument("--min_count_gene_source", type=float, default=0)
    p.add_argument("--min_count_cell_source", type=float, default=0)
    p.add_argument("--min_density_gene_source", type=float, default=0.05)
    p.add_argument("--min_density_cell_source", type=float, default=0.1)
    p.add_argument("--normalization_source", type=str, default="sqrt")

    p.add_argument("--min_count_gene_target", type=float, default=0)
    p.add_argument("--min_count_cell_target", type=float, default=0)
    p.add_argument("--gene_selection_count_source", type=int, default=1000)
    p.add_argument("--gene_selection_count_target", type=int, default=2000)
    p.add_argument("--clip_outlier_target", action="store_true", default=True)
    p.add_argument("--min_density_gene_target", type=float, default=0.05)
    p.add_argument("--min_density_cell_target", type=float, default=0.1)
    p.add_argument("--normalization_target", type=str, default="sqrt")

    p.add_argument("--data_dir", type=str, default=None, help="paired dataset root dir")

    # Model options
    p.add_argument("--enc_type_source", type=str, default="1d_simple")
    p.add_argument("--enc_type_target", type=str, default="1d_simple")

    # 1D model params
    p.add_argument("--enc_features_source", type=int, nargs="+", default=[2048, 1024, 512])
    p.add_argument("--dec_features_source", type=int, nargs="+", default=[512, 1024, 2048])
    p.add_argument("--disc_features_source", type=int, nargs="+", default=[256, 128, 128, 64])
    p.add_argument("--latent_dim_source", type=int, default=256)

    p.add_argument("--enc_features_target", type=int, nargs="+", default=[2048, 1024, 512])
    p.add_argument("--dec_features_target", type=int, nargs="+", default=[512, 1024, 2048])
    p.add_argument("--disc_features_target", type=int, nargs="+", default=[256, 128, 128, 64])
    p.add_argument("--latent_dim_target", type=int, default=512)

    p.add_argument("--trans_features_s2t", type=int, nargs="+", default=[512, 512, 512, 1024, 1024])
    p.add_argument("--trans_features_t2s", type=int, nargs="+", default=[512, 256, 256, 128, 128])

    return p

parser = get_parser_with_defaults()

args = parser.parse_args([])


if args.data_dir is None:
    args.data_dir = str((Path(os.getcwd()) / "../beanfur/gene_exp/gene_exp_private/data/paired_datasets").resolve())

opts = {
    "train_opt": {
        "loss_type_enc_dec": args.loss_type_enc_dec,
        "loss_type_cyc": args.loss_type_cyc,
        "loss_type_id": args.loss_type_id,
        "loss_type_disc": args.loss_type_disc,
        "lambda_cyc": args.lambda_cyc,
        "lambda_id": args.lambda_id,
        "lambda_adv": args.lambda_adv,
        "lambda_disc": args.lambda_disc,
        "lambda_gp": args.lambda_gp,
        "interval_enc_dec": args.interval_enc_dec,
        "interval_id": args.interval_id,
        "interval_cyc": args.interval_cyc,
        "interval_adv": args.interval_adv,
        "interval_disc": args.interval_disc,
        "batch_size": args.batch_size,
        "epochs_enc_dec": args.epochs_enc_dec,
        "fix_weight_enc_dec": args.fix_weight_enc_dec,
        "epochs": args.epochs,
        "cv_gene": args.cv_gene,
        "cv_sample": args.cv_sample,
        "target_fold_gene": args.target_fold_gene,
        "target_fold_sample": args.target_fold_sample,
        "device": args.device,
        "num_workers": args.num_workers,
        "log_dir": args.log_dir,
        "r1_every": args.r1_every,
        "lr_disc": args.lr_disc,
        "lr_trans": args.lr_trans,
        "seed": args.seed,
        "eval_every": args.eval_every,
        "eval_max_cells": args.eval_max_cells,
        "early_stop_patience": args.early_stop_patience,
        "early_stop_min_delta": args.early_stop_min_delta,
        "use_amp": args.use_amp,
    },
    "data_opt": {
        "domain_source": args.domain_source,
        "domain_target": args.domain_target,
        "data_dir": args.data_dir,
        "clip_outlier_source": args.clip_outlier_source,
        "min_count_gene_source": args.min_count_gene_source,
        "min_count_cell_source": args.min_count_cell_source,
        "min_density_gene_source": args.min_density_gene_source,
        "min_density_cell_source": args.min_density_cell_source,
        "normalization_source": args.normalization_source,
        "min_count_gene_target": args.min_count_gene_target,
        "min_count_cell_target": args.min_count_cell_target,
        "gene_selection_count_source": args.gene_selection_count_source,
        "gene_selection_count_target": args.gene_selection_count_target,
        "clip_outlier_target": args.clip_outlier_target,
        "min_density_gene_target": args.min_density_gene_target,
        "min_density_cell_target": args.min_density_cell_target,
        "normalization_target": args.normalization_target,
    },
    "model_opt": {
        "enc_type_source": args.enc_type_source,
        "enc_type_target": args.enc_type_target,
        "enc_features_source": args.enc_features_source,
        "dec_features_source": args.dec_features_source,
        "disc_features_source": args.disc_features_source,
        "latent_dim_source": args.latent_dim_source,
        "enc_features_target": args.enc_features_target,
        "dec_features_target": args.dec_features_target,
        "disc_features_target": args.disc_features_target,
        "latent_dim_target": args.latent_dim_target,
        "trans_features_s2t": args.trans_features_s2t,
        "trans_features_t2s": args.trans_features_t2s,
    }
}

train_opt = opts["train_opt"]
data_opt  = opts["data_opt"]
model_opt = opts["model_opt"]

os.makedirs(train_opt["log_dir"], exist_ok=True)

print("domains:", data_opt["domain_source"], "->", data_opt["domain_target"])
print("enc types:", model_opt["enc_type_source"], "/", model_opt["enc_type_target"])


domains: nanostring -> gse
enc types: 1d_simple / 1d_simple


In [4]:
# Data loader helper
def make_loader(ds, batch_size, shuffle, drop_last, num_workers, seed=1234, collate_fn=None):
    import random as _random

    def seed_worker(worker_id):
        worker_seed = (seed + worker_id) % (2**32)
        np.random.seed(worker_seed)
        _random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    g = torch.Generator()
    g.manual_seed(int(seed))

    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
        worker_init_fn=seed_worker if num_workers > 0 else None,
        generator=g if shuffle else None,
    )
    if num_workers > 0:
        kwargs["prefetch_factor"] = 2
    if collate_fn is not None:
        kwargs["collate_fn"] = collate_fn
    return torch.utils.data.DataLoader(ds, **kwargs)


In [5]:
# Load and process dataframe

print("Loading source/target dataframes...")
df_source = load_dataframe(data_opt["domain_source"], data_opt["data_dir"])
df_target = load_dataframe(data_opt["domain_target"], data_opt["data_dir"])

genes_to_keep = list(set(df_source.columns).intersection(set(df_target.columns)))

df_source = process_dataframe(
    df_source,
    data_opt["min_count_gene_source"], data_opt["min_count_cell_source"],
    data_opt["min_density_gene_source"], data_opt["min_density_cell_source"],
    data_opt["gene_selection_count_source"], data_opt["clip_outlier_source"],
    data_opt["normalization_source"],
    genes_to_keep=genes_to_keep
)

df_target = process_dataframe(
    df_target,
    data_opt["min_count_gene_target"], data_opt["min_count_cell_target"],
    data_opt["min_density_gene_target"], data_opt["min_density_cell_target"],
    data_opt["gene_selection_count_target"], data_opt["clip_outlier_target"],
    data_opt["normalization_target"],
    genes_to_keep=genes_to_keep
)

source_input_type = "1d" if model_opt["enc_type_source"].split("_")[0] == "1d" else "composite"

# Sort genes
df_source = df_source.loc[:, sorted(df_source.columns)]
df_target = df_target.loc[:, sorted(df_target.columns)]

gene_exp_union = sorted(set(df_source.columns).union(set(df_target.columns)))
gene_exp_inter = sorted(set(df_source.columns).intersection(set(df_target.columns)))
gene_exp_input_target = df_target.columns.tolist()

print("Processed base dataframe loaded")
print("source dataframe shape:", df_source.shape)
print("target dataframe shape:", df_target.shape)
print("# union genes:", len(gene_exp_union))
print("# intersect genes:", len(gene_exp_inter))
print("source_input_type:", source_input_type)


Loading source/target dataframes...
Processed base dataframe loaded
source dataframe shape: (46496, 960)
target dataframe shape: (34967, 2000)
# union genes: 2472
# intersect genes: 488
source_input_type: 1d


In [6]:
# Create CV folds
cv_gene   = int(train_opt["cv_gene"])
cv_sample = int(train_opt["cv_sample"])

list_fold_gene   = [i for i in range(cv_gene)]   if train_opt.get("target_fold_gene")   is None else [int(train_opt["target_fold_gene"])]
list_fold_sample = [i for i in range(cv_sample)] if train_opt.get("target_fold_sample") is None else [int(train_opt["target_fold_sample"])]

fold_gene_path   = os.path.join(train_opt["log_dir"], "fold_gene_assignments.csv")
fold_sample_path = os.path.join(train_opt["log_dir"], "fold_sample_assignments.csv")

if os.path.exists(fold_gene_path):
    fg = pd.read_csv(fold_gene_path)
    gene2fold = dict(zip(fg["gene"], fg["fold"]))
else:
    gene2fold = {g: (i % cv_gene) for i, g in enumerate(gene_exp_inter)}
    pd.DataFrame({"gene": list(gene2fold.keys()), "fold": list(gene2fold.values())}).to_csv(fold_gene_path, index=False)

if os.path.exists(fold_sample_path):
    fs = pd.read_csv(fold_sample_path)
    cell2fold = dict(zip(fs["cell_id"], fs["fold"]))
else:
    cell2fold = {cid: (i % cv_sample) for i, cid in enumerate(df_source.index.tolist())}
    pd.DataFrame({"cell_id": list(cell2fold.keys()), "fold": list(cell2fold.values())}).to_csv(fold_sample_path, index=False)

opts["gene_names"] = {
    "source": df_source.columns.tolist(),
    "target": df_target.columns.tolist(),
    "intersection": gene_exp_inter,
    "fold": {fold: [g for g in gene_exp_inter if gene2fold[g] == fold] for fold in list_fold_gene},
}
opts["cell_ids"] = {
    fold: [cid for cid in df_source.index.tolist() if cell2fold[cid] == fold]
    for fold in list_fold_sample
}

print("Fold assignment files:")
print(" ", fold_gene_path)
print(" ", fold_sample_path)


Fold assignment files:
  results/nano2gse/fold_gene_assignments.csv
  results/nano2gse/fold_sample_assignments.csv


In [7]:
# Build dataloader and train
start_time = time.time()

for fold_gene in list_fold_gene:
    for fold_sample in list_fold_sample:
        print(f"Running fold_gene {fold_gene+1}/{len(list_fold_gene)} | fold_sample {fold_sample+1}/{len(list_fold_sample)}")

        # Gene split
        gene_exp_val  = [g for i, g in enumerate(gene_exp_inter) if (i % cv_gene) != fold_gene]
        gene_exp_test = [g for i, g in enumerate(gene_exp_inter) if (i % cv_gene) == fold_gene]
        gene_exp_input_source = [c for c in df_source.columns if c not in gene_exp_test]

        # Sample split
        if cv_sample == 1:
            list_train_cell_id_source = df_source.index.tolist()
            list_val_cell_id_source   = df_source.index.tolist()
            list_test_cell_id_source  = df_source.index.tolist()
        else:
            list_dev_cell_id_source = [cid for i, cid in enumerate(df_source.index) if (i % cv_sample) != fold_sample]
            list_train_cell_id_source = [cid for i, cid in enumerate(list_dev_cell_id_source) if i % 10 != 0]
            list_val_cell_id_source   = [cid for i, cid in enumerate(list_dev_cell_id_source) if i % 10 == 0]
            list_test_cell_id_source  = [cid for i, cid in enumerate(df_source.index) if (i % cv_sample) == fold_sample]

        opts["exp_setting"] = {
            "fold_gene": fold_gene,
            "fold_sample": fold_sample,
            "cv_gene": cv_gene,
            "cv_sample": cv_sample,
            "cell_id": {
                "train": list_train_cell_id_source,
                "val":   list_val_cell_id_source,
                "test":  list_test_cell_id_source,
            },
            "gene_names": {
                "val": gene_exp_val,
                "test": gene_exp_test,
                "source": df_source.columns.tolist(),
                "source_input": gene_exp_input_source,
                "target": df_target.columns.tolist(),
            },
        }

        # DataFrames for this fold
        if cv_sample == 1:
            dict_df = {
                "source": {
                    "train": df_source.loc[list_train_cell_id_source],
                    "val":   df_source.loc[list_val_cell_id_source],
                },
                "target": {"train": df_target},
            }
        else:
            dict_df = {
                "source": {
                    "train": df_source.loc[list_train_cell_id_source],
                    "val":   df_source.loc[list_val_cell_id_source],
                    "test":  df_source.loc[list_test_cell_id_source],
                },
                "target": {"train": df_target},
            }

        # -------------------------
        # Build datasets
        # -------------------------
        dict_ds = {
            "target": {"train": ExpDataset(df_target, gene_exp_input_target)}
        }

        if source_input_type == "1d":
            dict_ds["source"] = {
                split: ExpDataset(dict_df["source"][split], gene_exp_input_source, gene_exp_test)
                for split in dict_df["source"].keys()
            }
        else:
            if data_opt["domain_source"] != "nanostring":
                raise ValueError("Composite input path here is configured for 'nanostring' like the original script.")

            image_dir = os.path.join(data_opt["data_dir"], "nanostring", "image")
            label_dir = os.path.join(data_opt["data_dir"], "nanostring", "image_label")
            df_meta = pd.read_csv(os.path.join(data_opt["data_dir"], "nanostring", "Lung9_Rep1_metadata_file.csv"))

            dict_image_data = {
                "image": {
                    fov: tif.imread(glob.glob(os.path.join(image_dir, f"*F{str(fov).zfill(3)}_Z003*"))[0])
                    for fov in range(1, 21)
                },
                "label": {
                    fov: tif.imread(os.path.join(label_dir, f"CellLabels_F{str(fov).zfill(3)}.tif"))
                    for fov in range(1, 21)
                },
            }

            dict_image_data["meta"] = {}
            for index in df_source.index:
                fov = int(index.split("-")[0].split("_")[-1])
                cell_id = int(index.split("-")[1])
                cx, cy, width, height = df_meta.loc[
                    (df_meta["fov"] == fov) & (df_meta["cell_ID"] == cell_id),
                    ["CenterX_local_px", "CenterY_local_px", "Width", "Height"],
                ].values[0]
                dict_image_data["meta"][index] = dict(cx=float(cx), cy=float(cy), width=float(width), height=float(height))

            sampled_indices = []
            sample_size = len(dict_df["source"]["train"])
            while len(sampled_indices) < sample_size:
                idx = np.random.choice(dict_df["source"]["train"].index)
                if all(
                    np.linalg.norm(df_source.loc[idx][["x", "y"]].values - df_source.loc[sidx][["x", "y"]].values) > 5
                    for sidx in sampled_indices
                ):
                    sampled_indices.append(idx)

            target_size = tuple(data_opt["input_size_image"])

            dict_ds["source"] = {
                split: CompositeDataset(
                    dict_df["source"][split].loc[sampled_indices],
                    dict_image_data,
                    gene_exp_input_source,
                    gene_exp_test,
                    target_size=target_size,
                )
                for split in dict_df["source"].keys()
            }

        # -------------------------
        # Build dataloaders
        # -------------------------
        dict_dl = {}
        dict_dl["target"] = {
            "train": make_loader(
                dict_ds["target"]["train"],
                batch_size=train_opt["batch_size"],
                shuffle=True,
                drop_last=True,
                num_workers=train_opt["num_workers"],
                seed=train_opt["seed"],
            )
        }

        dict_dl["source"] = {}
        for split in dict_ds["source"].keys():
            shuffle = (split == "train")
            drop_last = (split == "train")

            if source_input_type == "1d":
                dict_dl["source"][split] = make_loader(
                    dict_ds["source"][split],
                    batch_size=train_opt["batch_size"],
                    shuffle=shuffle,
                    drop_last=drop_last,
                    num_workers=train_opt["num_workers"],
                    seed=train_opt["seed"],
                )
            else:
                dict_dl["source"][split] = make_loader(
                    dict_ds["source"][split],
                    batch_size=train_opt["batch_size"],
                    shuffle=shuffle,
                    drop_last=drop_last,
                    num_workers=train_opt["num_workers"],
                    seed=train_opt["seed"],
                    collate_fn=composite_collate_fn,
                )

        trainer = define_trainer(len(gene_exp_input_source), len(gene_exp_input_target), opts, df_source)
        # Train encoder decoder
        trainer.train_enc_dec(dict_dl)
        # Train translator
        trainer.train(dict_dl)

elapsed = time.time() - start_time
print(f"CV training finished in {elapsed/60:.2f} minutes")

Running fold_gene 1/5 | fold_sample 1/1
Train Encoder Decoder for source domain.
[enc_dec] best at epoch 1 val_loss=0.1814
[enc_dec] best at epoch 2 val_loss=0.1808
[enc_dec] best at epoch 3 val_loss=0.1713
[enc_dec] best at epoch 4 val_loss=0.1708
[enc_dec] best at epoch 5 val_loss=0.1669
[enc_dec] best at epoch 6 val_loss=0.1657
[enc_dec] best at epoch 7 val_loss=0.164
[enc_dec] best at epoch 10 val_loss=0.1625
[enc_dec] best at epoch 11 val_loss=0.1615
[enc_dec] best at epoch 13 val_loss=0.1607
[enc_dec] best at epoch 14 val_loss=0.1585
[enc_dec] best at epoch 15 val_loss=0.1585
[enc_dec] best at epoch 16 val_loss=0.1583
[enc_dec] best at epoch 17 val_loss=0.1571
[enc_dec] best at epoch 19 val_loss=0.1564
[enc_dec] best at epoch 21 val_loss=0.1552
[enc_dec] best at epoch 23 val_loss=0.1552
[enc_dec] best at epoch 24 val_loss=0.1536
[enc_dec] best at epoch 25 val_loss=0.1534
[enc_dec] best at epoch 26 val_loss=0.1527
[enc_dec] best at epoch 27 val_loss=0.1521
[enc_dec] best at epoch 

  2%|██▏                                                                                                         | 1/50 [00:04<03:43,  4.55s/it]

[eval][epoch 1] split=  val  corr_val=0.2439  corr_test=0.2281
(epoch 1) do_eval=True  split_for_print=val  corr_val=0.2439  corr_test=0.2281  best_corr_val=0.2439 best_epoch=1


 10%|██████████▊                                                                                                 | 5/50 [00:21<03:22,  4.49s/it]

[eval][epoch 5] split=  val  corr_val=0.2404  corr_test=0.2196
(epoch 5) do_eval=True  split_for_print=val  corr_val=0.2404  corr_test=0.2196  best_corr_val=0.2439 best_epoch=1


...

 90%|████████████████████████████████████████████████████████████████████████████████████████████████▎          | 45/50 [03:28<00:23,  4.80s/it]

[eval][epoch 45] split=  val  corr_val=0.2300  corr_test=0.2104
(epoch 45) do_eval=True  split_for_print=val  corr_val=0.2300  corr_test=0.2104  best_corr_val=0.2449 best_epoch=1


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [03:50<00:00,  4.60s/it]

[eval][epoch 50] split=  val  corr_val=0.2322  corr_test=0.2120
(epoch 50) do_eval=True  split_for_print=val  corr_val=0.2322  corr_test=0.2120  best_corr_val=0.2449 best_epoch=1


CV training finished in 36.02 minutes


In [8]:
# Compute metrics for held-out folds

result_dir = train_opt["log_dir"]
os.makedirs(result_dir, exist_ok=True)

pd.DataFrame(gene_exp_inter, columns=["Common gene"]).to_csv(os.path.join(result_dir, "common_genes.csv"), index=False)

df_pred_parts = []
gene_fold = {}

for fold_gene in list_fold_gene:
    list_df = []
    for fold_sample in list_fold_sample:
        pkl_path = os.path.join(
            result_dir,
            f"fold_gene_{fold_gene}/fold_sample_{fold_sample}/predictions/best_pred_sample-test_gene-test.pkl"
        )
        df_temp = pd.read_pickle(pkl_path)

        cell_ids   = opts["cell_ids"][fold_sample]
        gene_names = opts["gene_names"]["fold"][fold_gene]
        list_df.append(df_temp.reindex(index=cell_ids, columns=gene_names))

    df_fold = pd.concat(list_df)
    df_pred_parts.append(df_fold)
    gene_fold.update({c: fold_gene for c in df_fold.columns})

df_pred = pd.concat(df_pred_parts, axis=1)

df_real = df_source.reindex(index=df_pred.index, columns=gene_exp_inter)
df_pred = df_pred.reindex(index=df_pred.index, columns=gene_exp_inter)

df_real.to_csv(os.path.join(result_dir, "real.csv"))
df_pred.to_csv(os.path.join(result_dir, "pred.csv"))

def pearson_vec(r, p):
    r = np.asarray(r, float); p = np.asarray(p, float)
    if np.std(r) == 0 or np.std(p) == 0:
        return np.nan
    return float(np.corrcoef(r, p)[0, 1])

def rmse_vec(r, p):
    r = np.asarray(r, float); p = np.asarray(p, float)
    r = (r - np.mean(r)) / (np.std(r) + 1e-8)
    p = (p - np.mean(p)) / (np.std(p) + 1e-8)
    return float(np.sqrt(((r - p) ** 2).mean()))

def compute_ssim_vec(x, y, C1=0.01, C2=0.03):
    x = np.asarray(x, float); y = np.asarray(y, float)
    x_scaled = (x - np.min(x)) / (np.max(x) - np.min(x) + 1e-8)
    y_scaled = (y - np.min(y)) / (np.max(y) - np.min(y) + 1e-8)
    ux, uy = np.mean(x_scaled), np.mean(y_scaled)
    var_x, var_y = np.var(x_scaled), np.var(y_scaled)
    cov_xy = np.cov(x_scaled, y_scaled)[0, 1]
    num = (2 * ux * uy + C1) * (2 * cov_xy + C2)
    den = (ux**2 + uy**2 + C1) * (var_x + var_y + C2)
    return float(num / den)

rows = []
for g in gene_exp_inter:
    r = df_real[g].values
    p = df_pred[g].values
    rows.append({
        "gene_name": g,
        "corr": pearson_vec(r, p),
        "rmse": rmse_vec(r, p),
        "ssim": compute_ssim_vec(r, p),
        "cv_fold": gene_fold[g],
    })

df_performance = pd.DataFrame(rows)
df_performance.to_csv(os.path.join(result_dir, "performance.csv"), index=False)

df_per_fold_mean = df_performance.groupby("cv_fold").mean(numeric_only=True)
final_average = df_per_fold_mean.mean(numeric_only=True)
pd.DataFrame(final_average).reset_index().rename(
    columns={0: "Final Average Value", "index": "Metric"}
)

print("Saved metrics to:", result_dir)
display(df_performance.head())

Saved metrics to: results/nano2gse


,gene_name,corr,rmse,ssim,cv_fold
0,ABL2,0.286417,1.194641,0.379009,0
1,ACE,0.172073,1.286800,0.381492,1
2,ACKR1,0.136452,1.314190,0.354319,2
3,ACTA2,0.229331,1.241506,0.435314,3
4,ACTG2,0.052568,1.376540,0.371972,4


In [9]:
# Train on all data

gene_exp_input_source = df_source.columns.tolist()
gene_exp_input_target = df_target.columns.tolist()

dict_ds_full = {
    "source": {"train": ExpDataset(df_source, gene_exp_input_source)},
    "target": {"train": ExpDataset(df_target, gene_exp_input_target)},
}

dict_dl_full = {
    "source": {
        "train": make_loader(dict_ds_full["source"]["train"], train_opt["batch_size"], True,  True,  train_opt["num_workers"], seed=train_opt["seed"]),
        "val":   make_loader(dict_ds_full["source"]["train"], train_opt["batch_size"], False, True,  train_opt["num_workers"], seed=train_opt["seed"]),
    },
    "target": {
        "train": make_loader(dict_ds_full["target"]["train"], train_opt["batch_size"], True,  True,  train_opt["num_workers"], seed=train_opt["seed"]),
        "val":   make_loader(dict_ds_full["target"]["train"], train_opt["batch_size"], False, True,  train_opt["num_workers"], seed=train_opt["seed"]),
    }
}

trainer_full = define_trainer(len(gene_exp_input_source), len(gene_exp_input_target), opts, df_source)

print("[Full] Training encoder-decoder...")
trainer_full.train_enc_dec(dict_dl_full)
print("[Full] Training translator...")
trainer_full.train(dict_dl_full)

print("[Full] Predicting imputed matrix for all cells & genes...")
trainer_full.enc_source.eval()
trainer_full.trans_s2t.eval()
trainer_full.dec_target.eval()

use_amp = bool(train_opt.get("use_amp", True))
device = trainer_full.device

all_cell_ids, all_preds = [], []
with torch.no_grad(), torch.amp.autocast(device_type="cuda", enabled=use_amp):
    for batch in dict_dl_full["source"]["train"]:
        x = batch["input"].to(device, non_blocking=True)
        pred = trainer_full.translate_s2t({"input": x})
        all_preds.append(pred.float().cpu().numpy())
        all_cell_ids.extend(batch["index"])

pred_matrix = np.concatenate(all_preds, axis=0)
df_full_pred = pd.DataFrame(pred_matrix, index=all_cell_ids, columns=gene_exp_input_target)

out_path = os.path.join(train_opt["log_dir"], "full_data_imputed.csv")
df_full_pred.to_csv(out_path)
print("[Full] Saved:", out_path)

[Full] Training encoder-decoder...
Train Encoder Decoder for source domain.
[enc_dec] best at epoch 1 val_loss=0.1803
[enc_dec] best at epoch 2 val_loss=0.1748
[enc_dec] best at epoch 3 val_loss=0.1741
[enc_dec] best at epoch 4 val_loss=0.1726
[enc_dec] best at epoch 5 val_loss=0.1701
[enc_dec] best at epoch 7 val_loss=0.1683
[enc_dec] best at epoch 8 val_loss=0.1662
[enc_dec] best at epoch 9 val_loss=0.1651
[enc_dec] best at epoch 11 val_loss=0.165
[enc_dec] best at epoch 12 val_loss=0.162
[enc_dec] best at epoch 14 val_loss=0.1616
[enc_dec] best at epoch 15 val_loss=0.1608
[enc_dec] best at epoch 16 val_loss=0.1598
[enc_dec] best at epoch 18 val_loss=0.1589
[enc_dec] best at epoch 19 val_loss=0.1588
[enc_dec] best at epoch 21 val_loss=0.1584
[enc_dec] best at epoch 24 val_loss=0.1573
[enc_dec] best at epoch 25 val_loss=0.1568
[enc_dec] best at epoch 26 val_loss=0.1562
[enc_dec] best at epoch 27 val_loss=0.1554
[enc_dec] best at epoch 28 val_loss=0.1553
[enc_dec] best at epoch 30 val_

  2%|██▏                                                                                                         | 1/50 [00:05<04:09,  5.10s/it]

[eval][epoch 1] split=  val  corr_val=0.2448  corr_test=0.2439
(epoch 1) do_eval=True  split_for_print=val  corr_val=0.2448  corr_test=0.2439  best_corr_val=0.2448 best_epoch=1


 10%|██████████▊                                                                                                 | 5/50 [00:25<04:05,  5.46s/it]

[eval][epoch 5] split=  val  corr_val=0.2446  corr_test=0.2387
(epoch 5) do_eval=True  split_for_print=val  corr_val=0.2446  corr_test=0.2387  best_corr_val=0.2448 best_epoch=1


...

 90%|████████████████████████████████████████████████████████████████████████████████████████████████▎          | 45/50 [03:29<00:24,  4.86s/it]

[eval][epoch 45] split=  val  corr_val=0.2295  corr_test=0.2323
(epoch 45) do_eval=True  split_for_print=val  corr_val=0.2295  corr_test=0.2323  best_corr_val=0.2448 best_epoch=1


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [03:55<00:00,  4.70s/it]

[eval][epoch 50] split=  val  corr_val=0.2306  corr_test=0.2342
(epoch 50) do_eval=True  split_for_print=val  corr_val=0.2306  corr_test=0.2342  best_corr_val=0.2448 best_epoch=1


[Full] Predicting imputed matrix for all cells & genes...
[Full] Saved: results/nano2gse/full_data_imputed.csv
